In [3]:
import numpy as np
import random

## State and Action Definition

### State (Hybrid State)
State is defined as:
- Predicted traffic cluster (0–3)
- Queue level (Low, Medium, High → 0,1,2)

Total states = 4 × 3 = 12

### Action Space
0 → Keep timings unchanged  
1 → Increase green on main road  
2 → Increase green on side road  
3 → Increase total cycle time  
4 → Decrease total cycle time


In [4]:
class HybridTrafficAgent:
    def __init__(
        self,
        num_clusters=4,
        num_queue_levels=3,
        num_actions=5,
        alpha=0.1,
        gamma=0.9,
        epsilon=0.2,
        history_size=3,
        use_prediction=True
    ):
        self.num_clusters = num_clusters
        self.num_queue_levels = num_queue_levels
        self.num_actions = num_actions

        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        self.history_size = history_size
        self.use_prediction = use_prediction

        self.history = []

        self.q_table = np.zeros(
            (num_clusters, num_queue_levels, num_actions)
        )

    def observe(self, state):
        self.history.append(state)
        if len(self.history) > self.history_size:
            self.history.pop(0)

    def predict_cluster(self):
        if not self.use_prediction or len(self.history) < 2:
            return self.history[-1]["cluster"]

        volumes = []
        for s in self.history:
            volumes.append(s["vehicle_count"])

        trend = volumes[-1] - volumes[0]
        current_cluster = self.history[-1]["cluster"]

        if trend > 0:
            return min(current_cluster + 1, self.num_clusters - 1)
        elif trend < 0:
            return max(current_cluster - 1, 0)
        else:
            return current_cluster

    def discretize_queue(self, queue_length):
        if queue_length < 10:
            return 0
        elif queue_length < 25:
            return 1
        else:
            return 2

    def select_action(self, predicted_cluster, queue_level):
        if random.random() < self.epsilon:
            return random.randint(0, self.num_actions - 1)

        return int(
            np.argmax(self.q_table[predicted_cluster, queue_level])
        )

    def update_q_table(self, state, action, reward, next_state):
        c, q = state
        c_next, q_next = next_state

        best_next_value = np.max(
            self.q_table[c_next, q_next]
        )

        self.q_table[c, q, action] += self.alpha * (
            reward
            + self.gamma * best_next_value
            - self.q_table[c, q, action]
        )


In [5]:
agent = HybridTrafficAgent(
    alpha=0.1,
    gamma=0.9,
    epsilon=0.2,
    history_size=3,
    use_prediction=True
)

agent.q_table.shape


(4, 3, 5)

In [6]:
example_state = {
    "cluster": 2,
    "vehicle_count": 35,
    "queue_length": 18,
    "avg_speed": 22,
    "incident": 0
}

agent.observe(example_state)


In [7]:
predicted_cluster = agent.predict_cluster()
queue_level = agent.discretize_queue(example_state["queue_length"])

predicted_cluster, queue_level


(2, 1)

In [8]:
action = agent.select_action(predicted_cluster, queue_level)
action


0

In [ ]:
# reward calculation    
vehicles_passed = 12
avg_waiting_time = 25
queue_length = example_state["queue_length"]

reward = (
    vehicles_passed
    - 0.5 * avg_waiting_time
    - 0.2 * queue_length
)

reward


-4.1

In [10]:
current_state = (predicted_cluster, queue_level)

next_cluster = predicted_cluster
next_queue_length = 12
next_queue_level = agent.discretize_queue(next_queue_length)

next_state = (next_cluster, next_queue_level)

agent.update_q_table(
    current_state,
    action,
    reward,
    next_state
)


In [11]:
agent.q_table[predicted_cluster]


array([[ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ],
       [-0.41,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.  ]])

In [12]:
agent.use_prediction = False
print("Prediction disabled for ablation study.")


Prediction disabled for ablation study.
